# 4. Baseline dataset: what was known at time zero

**What it does:** builds one row per patient with everything known at presentation, in three groups
(`baseline_dictionary.csv` lists every variable and its group):

| Group | Contents | Used for |
|---|---|---|
| **A clustering inputs** | CT features from the NLP, vitals, labs, age, sex, comorbidities and Charlson, anticoagulation evidence, mechanism, other injuries, presenting symptoms, CTAS, mLAPS | defining phenotypes |
| **B equity** | neighbourhood marginalization (Ontario Marginalization Index), rurality | does the same phenotype get the same care? |
| **C care context** | hospital, teaching / non-teaching, year, flags, when each value was measured | adjustment |

Labs and vitals are the **value closest to time zero** (labs 24 h before to 12 h after; vitals
6 h either side), taken from the raw lab / vitals tables using GEMINI's standard concepts first.
Comorbidities ignore type-2 (in-hospital) diagnoses. Medications: orders from the start of the
episode to 24 h after time zero.

**Output:** `patient_level/baseline.csv`; aggregate `baseline_dictionary.csv` and three check tables.
**Time:** several minutes (reads the 11-million-row lab table and the vitals table in batches).

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Run

In [ ]:
%run ../03_baseline.py

## Results

In [ ]:
d = show("baseline_dictionary.csv", rows=10, note="every baseline variable with its group")
if d is not None:
    display(d.group.value_counts().rename("variables").to_frame())

In [ ]:
show("check_lab_vital_timing.csv", note="coverage of each lab and vital");

**Look for:** `pct_patients` = share of patients who have that value. **GCS** decides whether GCS can
be a clustering input. A large `outside_window` count means values exist but were measured too long
after arrival; many `implausible` values usually mean a unit problem.

In [ ]:
names = show("check_lab_vital_names.csv", rows=0)
if names is not None:
    for v, g in names.groupby("variable"):
        display(Markdown(f"**{v}**")); display(g.head(8))

**Look for:** each variable built from the right test (e.g. platelets not 'mean platelet volume') and consistent units.

In [ ]:
show("check_med_names.csv", rows=60, note="drug names matched to each medication group");

**Look for:** wrong drugs in a group (e.g. heparin flush under anticoagulants).

## Share

In [ ]:
share(["baseline_dictionary.csv", "check_lab_vital_timing.csv", "check_lab_vital_names.csv", "check_med_names.csv"])